<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_evaluation/stage_07_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07 – Model Evaluation**


# **Preparación de entorno**

## **1. Imports**

In [3]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-11 22:59:25,598 | INFO | Environment initialized


## **2. Acceso a drive**

In [4]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

Mounted at /content/drive


2026-04-11 22:59:51,483 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


## **3. Carga de métricas**

In [5]:
from pathlib import Path
import pandas as pd

def load_all_classification_metrics(
    *,
    base_dir: Path,
    pattern: str = "classification_*_metrics.parquet",
    verbose: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Carga todos los archivos parquet de métricas de clasificación.

    Retorna:
        dict {model_name: DataFrame}
    """

    paths = list(base_dir.glob(pattern))
    results = {}

    if verbose:
        print(f"[INFO] Encontrados {len(paths)} archivos")

    for path in paths:
        name = path.stem.replace("classification_", "").replace("_metrics", "")

        if verbose:
            print(f"[LOAD] {name} -> {path}")

        df = pd.read_parquet(path)
        results[name] = df

    return results

In [6]:
def concat_metrics(metrics_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    df_all = []

    for name, df in metrics_dict.items():
        df_copy = df.copy()
        df_copy["model"] = name
        df_all.append(df_copy)

    return pd.concat(df_all, ignore_index=True)

In [7]:
metrics_dir = DRIVE_DIR / "metrics/classification_metrics"

metrics_dict = load_all_classification_metrics(base_dir=metrics_dir)

# Ejemplo:
#metrics_dict["gru"].head()
#metrics_dict["lightgbm_balanced"].head()

df_all = pd.concat(
    [df.assign(model=name) for name, df in metrics_dict.items()],
    ignore_index=True
)

[INFO] Encontrados 14 archivos
[LOAD] logistic_regression -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_logistic_regression_metrics.parquet
[LOAD] logistic_regression_balanced -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_logistic_regression_balanced_metrics.parquet
[LOAD] random_forest -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_random_forest_metrics.parquet
[LOAD] random_forest_balanced -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_random_forest_balanced_metrics.parquet
[LOAD] xgboost -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_xgboost_metrics.parquet
[LOAD] xgboost_balanced -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_xgboost_balanced_metrics.parquet
[LOAD] lightgbm -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classi

# PLAN DE SELECCIÓN Y TUNING (SEQ2ONE T2)



## **1. Consolidación de resultados**

**Objetivo:** Tener una base única para análisis.

* Unificar todos los parquets en un solo DataFrame (`df_all`)
* Verificar columnas, tipos y consistencia
* Confirmar:

  * splits: train / valid / test
  * targets: 90 / 120
  * ventanas: L ∈ {30, 60, 90, 120, 180}

---

## **2. Definición de métricas de decisión**

**Objetivo:** Establecer criterio correcto (evitar conclusiones erróneas).

* Métrica principal:
  → `bal_acc_gain_vs_naive`

* Métricas de apoyo:
  → `balanced_accuracy`, `f1_macro`

* Regla:

  * modelo útil → gain > 0
  * modelo fuerte → gain > 0.03–0.05

---

## **3. Filtro de calidad (anti-ruido)**

**Objetivo:** Eliminar configuraciones no robustas.

* Mantener solo:

  * `split = test`
  * gain > umbral mínimo
* Validar consistencia:

  * comparar con `valid`
  * eliminar overfitting (test ≫ valid)

---

## **4. Ranking por modelo (nivel arquitectura)**

**Objetivo:** Identificar qué tipo de modelo funciona.

* Agrupar por `model`
* Promediar métricas
* Resultado:

  * ranking global de modelos

👉 Decisión:

* elegir 3 familias de modelos

---

## **5. Análisis por window_size (L)**

**Objetivo:** Encontrar escala temporal óptima.

* Agrupar por `window_size`
* Evaluar performance promedio

👉 Resultado:

* identificar si:

  * ventanas cortas → ruido
  * ventanas largas → señal

---

## **6. Análisis por target (90 vs 120)**

**Objetivo:** Detectar horizonte más predecible.

* Comparar:

  * `t2_dir_thr_90`
  * `t2_dir_thr_120`

👉 Resultado:

* cuál tiene mayor separabilidad

---

## **7. Análisis conjunto (modelo + L + target)**

**Objetivo:** Encontrar configuraciones óptimas reales.

* Ranking por combinación:

  * model + window_size + target
* Top N configuraciones

👉 Resultado:

* mejores setups concretos

---

## **8. Selección final de candidatos**

**Objetivo:** Reducir el espacio de búsqueda.

Elegir:

### ✔ 3 modelos

* 1 deep learning (ej: transformer)
* 1 ensemble (rf / xgb)
* 1 baseline (logistic)

### ✔ 2–3 ventanas

* las mejores según análisis

### ✔ 1–2 targets

* los más estables

---

## **9. Definición del espacio de tuning**

**Objetivo:** Preparar optimización eficiente.

Para cada modelo:

* definir hiperparámetros clave
* limitar rango (no búsqueda infinita)

Ejemplo:

* RF → depth, n_estimators
* XGB → learning_rate, max_depth
* Transformer → d_model, heads, dropout

---

## **10. Ejecución de tuning**

**Objetivo:** Optimizar performance real.

* usar:

  * valid para tuning
  * test solo para evaluación final
* mantener:

  * mismo pipeline
  * mismo split

---

## **11. Evaluación final**

**Objetivo:** Validar modelo ganador.

* comparar:

  * vs naive
  * vs baseline
* revisar:

  * estabilidad entre splits
  * consistencia por régimen (opcional)

---

## **12. Conclusión y cierre del stage**

**Objetivo:** Formalizar resultados.

* modelo ganador
* configuración óptima
* interpretación:

  * qué aprendió el modelo
  * qué señal existe en los datos

---

# 🔹 RESUMEN CLAVE

El flujo es:

```
Resultados → Filtrado → Ranking → Selección → Tuning → Validación
```

---

Si quieres, en el siguiente paso ejecutamos juntos el **punto 4 (ranking real de tus modelos)** y te doy directamente el shortlist óptimo.


# **1. Consolidación de resultados**

**Objetivo:** Tener una base única para análisis.

* Unificar todos los parquets en un solo DataFrame (`df_all`)
* Verificar columnas, tipos y consistencia
* Confirmar:

  * splits: train / valid / test
  * targets: 90 / 120
  * ventanas: L ∈ {30, 60, 90, 120, 180}

In [8]:
df_all

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,logistic_regression,test,30,t2_dir_thr_120,93990,0.345316,0.270370,0.438369,0.579987,0.383358,0.345316,0.333333,0.011983,120,none,logistic_regression
1,logistic_regression,valid,30,t2_dir_thr_120,93508,0.335323,0.283183,0.604690,0.720698,0.551519,0.335323,0.333333,0.001990,120,none,logistic_regression
2,logistic_regression,test,30,t2_dir_thr_90,93990,0.343799,0.268514,0.436526,0.577668,0.409567,0.343799,0.333333,0.010466,90,none,logistic_regression
3,logistic_regression,valid,30,t2_dir_thr_90,93508,0.335520,0.282303,0.596703,0.714420,0.497953,0.335520,0.333333,0.002187,90,none,logistic_regression
4,logistic_regression,test,60,t2_dir_thr_120,88140,0.344856,0.265729,0.420574,0.564772,0.428477,0.344856,0.333333,0.011523,120,none,logistic_regression
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,transformer_balanced,valid,120,t2_dir_thr_90,76048,0.416400,0.412012,0.596041,0.617229,0.434474,0.416400,0.333333,0.083067,90,balanced,transformer_balanced
276,transformer_balanced,test,180,t2_dir_thr_120,64740,0.445395,0.445841,0.514053,0.519741,0.447254,0.445395,0.333333,0.112062,120,balanced,transformer_balanced
277,transformer_balanced,valid,180,t2_dir_thr_120,64408,0.409654,0.410077,0.603517,0.618681,0.420697,0.409654,0.333333,0.076321,120,balanced,transformer_balanced
278,transformer_balanced,test,180,t2_dir_thr_90,64740,0.429520,0.425170,0.484447,0.492678,0.427173,0.429520,0.333333,0.096187,90,balanced,transformer_balanced


## 1.1. Verificación estructural

In [22]:
print('1. Validación estructural:\n')
df_all.info()
df_all.head()

1. Validación estructural:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model                    280 non-null    object 
 1   split                    280 non-null    object 
 2   window_size              280 non-null    int64  
 3   target                   280 non-null    object 
 4   n_samples                280 non-null    int64  
 5   balanced_accuracy        280 non-null    float64
 6   f1_macro                 280 non-null    float64
 7   f1_weighted              280 non-null    float64
 8   accuracy                 280 non-null    float64
 9   precision_macro          280 non-null    float64
 10  recall_macro             280 non-null    float64
 11  balanced_accuracy_naive  280 non-null    float64
 12  bal_acc_gain_vs_naive    280 non-null    float64
 13  horizon_min              280 non-null    int64  
 14

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,logistic_regression,test,30,t2_dir_thr_120,93990,0.345316,0.270370,0.438369,0.579987,0.383358,0.345316,0.333333,0.011983,120,none,logistic_regression
1,logistic_regression,valid,30,t2_dir_thr_120,93508,0.335323,0.283183,0.604690,0.720698,0.551519,0.335323,0.333333,0.001990,120,none,logistic_regression
2,logistic_regression,test,30,t2_dir_thr_90,93990,0.343799,0.268514,0.436526,0.577668,0.409567,0.343799,0.333333,0.010466,90,none,logistic_regression
3,logistic_regression,valid,30,t2_dir_thr_90,93508,0.335520,0.282303,0.596703,0.714420,0.497953,0.335520,0.333333,0.002187,90,none,logistic_regression
4,logistic_regression,test,60,t2_dir_thr_120,88140,0.344856,0.265729,0.420574,0.564772,0.428477,0.344856,0.333333,0.011523,120,none,logistic_regression


## 1.2. Verificación columnas clave

In [23]:
print('2. Verificación columnas clave:\n')


cols_expected = [
    "model", "split", "window_size", "target",
    "balanced_accuracy", "bal_acc_gain_vs_naive",
    "horizon_min", "class_weight_mode"
]

missing = [c for c in cols_expected if c not in df_all.columns]
print("Missing:", missing)

2. Verificación columnas clave:

Missing: []


## 1.3. Validar splits

In [24]:
print('3. Verificación splits:\n')
df_all["split"].value_counts()

3. Verificación splits:



,count
split,
test,140
valid,140


## 1.4. Validar targets

In [25]:
print('4. Validar targets:\n')
df_all["target"].value_counts()

4. Validar targets:



,count
target,
t2_dir_thr_120,140
t2_dir_thr_90,140


## 1.5. Validar ventanas (window_size)

In [27]:
print('5. Validar ventanas:\n')

sorted(df_all["window_size"].unique())

5. Validar ventanas:



[np.int64(30), np.int64(60), np.int64(90), np.int64(120), np.int64(180)]

## 1.6. Validar duplicados

In [26]:
print('6. Validar duplicados:\n')

dup_cols = ["model", "split", "window_size", "target"]
df_all.duplicated(subset=dup_cols).sum()

6. Validar duplicados:



np.int64(0)

## 1.7. Validar NaNs

In [29]:
print('7. Validar NaNs:\n')
df_all.isna().sum()

7. Validar NaNs:



,0
model,0
split,0
window_size,0
target,0
n_samples,0
balanced_accuracy,0
f1_macro,0
f1_weighted,0
accuracy,0
precision_macro,0


## 1.8. Validar baseline

In [28]:
print('8. Validar Baseline:\n')
df_all["balanced_accuracy_naive"].unique()

8. Validar Baseline:



array([0.33333333])

## 1.9. Validar consistencia gain

In [31]:
print('9. Validar consistencia gain:\n')
(df_all["balanced_accuracy"] - df_all["balanced_accuracy_naive"]
 - df_all["bal_acc_gain_vs_naive"]).abs().max()

9. Validar consistencia gain:



0.0

## 1.10. Validación final rápida

In [30]:
print('10. Validación final rápida:\n')

print("Shape:", df_all.shape)
print("Models:", df_all["model"].nunique())
print("Configs:", df_all.groupby(["model","window_size","target"]).ngroups)

10. Validación final rápida:

Shape: (280, 16)
Models: 14
Configs: 140


# **2. Definición de métricas de decisión**

**Objetivo:** Establecer criterio correcto (evitar conclusiones erróneas).

* Métrica principal:
  → `bal_acc_gain_vs_naive`

* Métricas de apoyo:
  → `balanced_accuracy`, `f1_macro`

* Regla:

  * modelo útil → gain > 0
  * modelo fuerte → gain > 0.03–0.05

## 2.1. Creación de flags de calidad

Qué representa cada flag:

- is_useful: supera al naive
- is_strong: señal ya interesante
- is_very_strong: señal claramente fuerte

In [32]:
df = df_all.copy()

# limpiar class_weight
df["class_weight_mode"] = df["class_weight_mode"].fillna("none")

# flags
df["is_useful"] = df["bal_acc_gain_vs_naive"] > 0
df["is_strong"] = df["bal_acc_gain_vs_naive"] > 0.03
df["is_very_strong"] = df["bal_acc_gain_vs_naive"] > 0.05

In [38]:
df[[
    "model", "split", "window_size", "target",
    "bal_acc_gain_vs_naive", "is_useful", "is_strong", "is_very_strong"
]].head()

,model,split,window_size,target,bal_acc_gain_vs_naive,is_useful,is_strong,is_very_strong
0,logistic_regression,test,30,t2_dir_thr_120,0.011983,True,False,False
1,logistic_regression,valid,30,t2_dir_thr_120,0.001990,True,False,False
2,logistic_regression,test,30,t2_dir_thr_90,0.010466,True,False,False
3,logistic_regression,valid,30,t2_dir_thr_90,0.002187,True,False,False
4,logistic_regression,test,60,t2_dir_thr_120,0.011523,True,False,False


## 2.2. Revisar distribución de señal

In [33]:
df.groupby("split")["bal_acc_gain_vs_naive"].describe()

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,140.0,0.046928,0.039747,0.006511,0.012122,0.017595,0.088136,0.115995
valid,140.0,0.035929,0.037644,0.000359,0.003245,0.007586,0.074878,0.103347


a) Diagnóstico de la distribución de señal

- Datos para el split test:

  * mean: 0.0469
  * median: 0.0176
  * percentil 75: 0.0881
  * máximo: 0.116

- Datos para el split valid:

  * mean: 0.0359
  * median: 0.0076
  * percentil 75: 0.0749
  * máximo: 0.103

---

b) Interpretación de la señal

1. Existe señal real en el dataset

    * El valor medio del gain es mayor que 0 tanto en test como en valid
    * Esto indica que los modelos superan al baseline naive
    * Por lo tanto, el dataset contiene alpha

2. La señal es altamente desigual

    * La mediana en test (0.0176) es baja
    * La media en test (0.0469) es significativamente mayor
    * Esto implica que la distribución está sesgada por pocos valores altos
    * Conclusión:

      * la mayoría de configuraciones tienen señal débil o nula
      * un subconjunto pequeño tiene señal fuerte

3. Existen configuraciones con señal muy fuerte

    * El máximo en test es aproximadamente 0.11
    * Este nivel de gain es alto para un problema T2 multiclase
    * Indica que ciertos modelos/configuraciones capturan bien la señal

4. Diferencia entre test y valid

    * mean test: 0.0469
    * mean valid: 0.0359
    * Existe una caída leve al pasar de test a valid
    * Interpretación:

      * hay algo de sobreajuste, pero no es severo
      * la señal se mantiene fuera de muestra

---

c) Conclusión técnica

El comportamiento observado corresponde a un caso de “sparse alpha”:

* la señal no está distribuida de forma uniforme
* depende de combinaciones específicas de:

  * modelo
  * window_size
  * target

---

d) Implicaciones para el análisis

* No es adecuado promediar resultados globalmente
* No es suficiente evaluar modelos por promedio general
* Es necesario identificar configuraciones específicas con alto rendimiento

---

e) Criterio de selección a partir de este punto

Se deben priorizar configuraciones con:

* gain > 0.05 → señal fuerte
* gain > 0.08 → señal muy fuerte

---

f) Estado del proceso

* El análisis de distribución de señal está completado
* Se confirma que:

  * existe señal
  * no es uniforme
  * hay configuraciones claramente superiores


## 2.3. Filtro de calidad (anti-ruido)


Objetivo: eliminar configuraciones no robustas y quedarnos solo con señal consistente.

a) Selección inicial

* Trabajar principalmente con `split = test`
* Aplicar un umbral mínimo de señal:

  * `bal_acc_gain_vs_naive > 0.01` (filtra ruido puro)

b) Identificación de señal relevante

* Marcar como candidatos:

  * gain > 0.03 → señal moderada
  * gain > 0.05 → señal fuerte

c) Validación de consistencia (clave)

* Comparar cada configuración en `test` contra `valid`
* Mantener solo configuraciones donde:

  * `gain_valid > 0`
* Opcional (más estricto):

  * evitar casos donde `test ≫ valid`

d) Eliminación de falsos positivos

* Descartar configuraciones donde:

  * buen resultado en test pero nulo o muy bajo en valid
* Esto reduce riesgo de overfitting

e) Resultado esperado

* Dataset reducido con:

  * configuraciones con señal real
  * consistentes entre valid y test
* Base confiable para ranking y selección final




In [39]:
# ================================
# 3. Filtro de calidad (anti-ruido)
# ================================

df_filt = df.copy()

# a) Separar splits
df_test = df_filt[df_filt["split"] == "test"].copy()
df_valid = df_filt[df_filt["split"] == "valid"].copy()

# b) Renombrar métricas para merge
df_test = df_test.rename(columns={
    "balanced_accuracy": "balanced_accuracy_test",
    "f1_macro": "f1_macro_test",
    "bal_acc_gain_vs_naive": "bal_acc_gain_vs_naive_test",
})

df_valid = df_valid.rename(columns={
    "balanced_accuracy": "balanced_accuracy_valid",
    "f1_macro": "f1_macro_valid",
    "bal_acc_gain_vs_naive": "bal_acc_gain_vs_naive_valid",
})

# c) Merge por configuración
merge_keys = ["model", "window_size", "target"]

df_filtered = df_test.merge(
    df_valid[
        merge_keys + [
            "balanced_accuracy_valid",
            "f1_macro_valid",
            "bal_acc_gain_vs_naive_valid",
        ]
    ],
    on=merge_keys,
    how="inner"
)

# d) Filtro anti-ruido
# - test con gain > 0.01
# - valid con gain > 0
df_filtered = df_filtered[
    (df_filtered["bal_acc_gain_vs_naive_test"] > 0.01) &
    (df_filtered["bal_acc_gain_vs_naive_valid"] > 0.00)
].copy()

# e) Gap entre test y valid para inspección
df_filtered["gain_gap_test_valid"] = (
    df_filtered["bal_acc_gain_vs_naive_test"] -
    df_filtered["bal_acc_gain_vs_naive_valid"]
)

# f) Flags opcionales de robustez
df_filtered["is_consistent"] = df_filtered["gain_gap_test_valid"] <= 0.03
df_filtered["is_very_consistent"] = df_filtered["gain_gap_test_valid"] <= 0.02

# g) Ordenar por mejor señal en test
df_filtered = df_filtered.sort_values(
    by="bal_acc_gain_vs_naive_test",
    ascending=False
).reset_index(drop=True)

print("Shape original:", df.shape)
print("Shape filtrado:", df_filtered.shape)

df_filtered.head(10)

Shape original: (280, 19)
Shape filtrado: (126, 25)


,model,split,window_size,target,n_samples,balanced_accuracy_test,f1_macro_test,f1_weighted,accuracy,precision_macro,...,family,is_useful,is_strong,is_very_strong,balanced_accuracy_valid,f1_macro_valid,bal_acc_gain_vs_naive_valid,gain_gap_test_valid,is_consistent,is_very_consistent
0,gru_balanced,test,30,t2_dir_thr_90,93990,0.449329,0.449092,0.541916,0.540951,0.448880,...,gru_balanced,True,True,True,0.436482,0.436358,0.103149,0.012847,True,True
1,transformer_balanced,test,180,t2_dir_thr_120,64740,0.445395,0.445841,0.514053,0.519741,0.447254,...,transformer_balanced,True,True,True,0.409654,0.410077,0.076321,0.035741,False,False
2,gru_balanced,test,60,t2_dir_thr_90,88140,0.445165,0.438910,0.525453,0.530633,0.441109,...,gru_balanced,True,True,True,0.436680,0.440747,0.103347,0.008485,True,True
3,lstm_balanced,test,30,t2_dir_thr_90,93990,0.443933,0.444121,0.541269,0.544207,0.444887,...,lstm_balanced,True,True,True,0.430917,0.434321,0.097584,0.013016,True,True
4,gru_balanced,test,180,t2_dir_thr_120,64740,0.440567,0.406979,0.490302,0.524668,0.439066,...,gru_balanced,True,True,True,0.417997,0.412663,0.084664,0.022570,True,False
5,gru_balanced,test,30,t2_dir_thr_120,93990,0.440140,0.440524,0.539329,0.546473,0.442635,...,gru_balanced,True,True,True,0.414784,0.418992,0.081450,0.025357,True,False
6,xgboost_balanced,test,30,t2_dir_thr_90,93990,0.438932,0.438846,0.535633,0.537695,0.439061,...,xgboost_balanced,True,True,True,0.432357,0.435098,0.099024,0.006575,True,True
7,lstm_balanced,test,30,t2_dir_thr_120,93990,0.438271,0.435395,0.532906,0.536759,0.437192,...,lstm_balanced,True,True,True,0.421880,0.425907,0.088547,0.016390,True,True
8,lstm_balanced,test,60,t2_dir_thr_90,88140,0.438210,0.437117,0.528868,0.539800,0.440513,...,lstm_balanced,True,True,True,0.419434,0.423268,0.086101,0.018776,True,True
9,lightgbm_balanced,test,30,t2_dir_thr_90,93990,0.437927,0.437943,0.534988,0.536823,0.438141,...,lightgbm_balanced,True,True,True,0.429279,0.431511,0.095946,0.008648,True,True


In [40]:
print("Configs filtradas:", len(df_filtered))
print("Consistentes (gap <= 0.03):", df_filtered["is_consistent"].sum())
print("Muy consistentes (gap <= 0.02):", df_filtered["is_very_consistent"].sum())

Configs filtradas: 126
Consistentes (gap <= 0.03): 122
Muy consistentes (gap <= 0.02): 106


In [36]:
df_test = df[df["split"] == "test"]

ranking = (
    df_test.groupby("model")["bal_acc_gain_vs_naive"]
    .mean()
    .sort_values(ascending=False)
)

ranking

,bal_acc_gain_vs_naive
model,
gru_balanced,0.101022
lstm_balanced,0.096814
transformer_balanced,0.091691
lightgbm_balanced,0.090509
xgboost_balanced,0.090000
logistic_regression_balanced,0.080493
random_forest,0.019448
gru,0.013813
xgboost,0.013154


In [37]:
df_test.sort_values("bal_acc_gain_vs_naive", ascending=False).head(10)

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family,is_useful,is_strong,is_very_strong
222,gru_balanced,test,30,t2_dir_thr_90,93990,0.449329,0.449092,0.541916,0.540951,0.448880,0.449329,0.333333,0.115995,90,balanced,gru_balanced,True,True,True
276,transformer_balanced,test,180,t2_dir_thr_120,64740,0.445395,0.445841,0.514053,0.519741,0.447254,0.445395,0.333333,0.112062,120,balanced,transformer_balanced,True,True,True
226,gru_balanced,test,60,t2_dir_thr_90,88140,0.445165,0.438910,0.525453,0.530633,0.441109,0.445165,0.333333,0.111832,90,balanced,gru_balanced,True,True,True
182,lstm_balanced,test,30,t2_dir_thr_90,93990,0.443933,0.444121,0.541269,0.544207,0.444887,0.443933,0.333333,0.110600,90,balanced,lstm_balanced,True,True,True
236,gru_balanced,test,180,t2_dir_thr_120,64740,0.440567,0.406979,0.490302,0.524668,0.439066,0.440567,0.333333,0.107234,120,balanced,gru_balanced,True,True,True
220,gru_balanced,test,30,t2_dir_thr_120,93990,0.440140,0.440524,0.539329,0.546473,0.442635,0.440140,0.333333,0.106807,120,balanced,gru_balanced,True,True,True
102,xgboost_balanced,test,30,t2_dir_thr_90,93990,0.438932,0.438846,0.535633,0.537695,0.439061,0.438932,0.333333,0.105599,90,balanced,xgboost_balanced,True,True,True
180,lstm_balanced,test,30,t2_dir_thr_120,93990,0.438271,0.435395,0.532906,0.536759,0.437192,0.438271,0.333333,0.104937,120,balanced,lstm_balanced,True,True,True
186,lstm_balanced,test,60,t2_dir_thr_90,88140,0.438210,0.437117,0.528868,0.539800,0.440513,0.438210,0.333333,0.104877,90,balanced,lstm_balanced,True,True,True
142,lightgbm_balanced,test,30,t2_dir_thr_90,93990,0.437927,0.437943,0.534988,0.536823,0.438141,0.437927,0.333333,0.104594,90,balanced,lightgbm_balanced,True,True,True
